# Research Methods: Coding Assignment 1

## Setup

The assignment provides eight invented households directly as Python lists. The four lists use the same household order. `income_text` is kept as supplied; a separate numeric income list will be created later as requested in Question 3(a).

For income `I`, consumption `C`, and household size `n`, a household is eligible when `I / n <= 600` EUR per person, including exactly EUR 600. Its transfer is `min(100 * n, max(C - I, 0))`: the consumption shortfall, capped at EUR 100 per person. An ineligible household receives zero. Consumption stays fixed.

Income after support is `Y = I + T`; saving before support is `S = I - C`; saving after support is `S_after = Y - C`. A saving share is saving divided by the corresponding income, and is undefined (`None`) when that income is zero. Negative saving is valid.

In [1]:
import math

In [2]:
household_ids = ["A01", "A02", "A03", "A04",
                 "A05", "A06", "A07", "A08"]

income_text = ["1200", "0", "1500", "700",
               "2400", "800", "0", "1800"]

consumptions = [1250, 150, 1600, 850, 2600, 950, 0, 1400]
household_sizes = [2, 1, 3, 1, 4, 2, 2, 3]

## Question 1: initial prediction

Before running the snippet, my predictions are:

```text
[1200, 800, 0, 600]
600 4
20%
```

In [3]:
incomes = [1000, 800, 0]
saved_incomes = incomes
first_share = (incomes[0] - 800) / incomes[0]
incomes[0] = 1200
saved_incomes.append(600)
print(incomes)
print(saved_incomes[-1], len(incomes))
print(f"{first_share:.0%}")

[1200, 800, 0, 600]
600 4
20%


### 1(a) Outputs and trace

The three output lines are:

```text
[1200, 800, 0, 600]
600 4
20%
```

Initially, `incomes` contains `[1000, 800, 0]`. The assignment `incomes[0] = 1200` changes its first element. Then `append(600)` adds a fourth element to that same list. The list therefore has length 4, and its last element is 600. `first_share` was calculated earlier as `(1000 - 800) / 1000 = 0.2`, so formatting it as a percentage with zero decimal places prints `20%`.

My initial predictions were correct.

### 1(b) Why the names and values behave this way

`saved_incomes = incomes` assigns a second name to the same list; it does not create another list. Thus, changing an element through `incomes` and appending through `saved_incomes` both affect the one shared list. By contrast, `first_share` stores the numeric result of the division at the moment that line runs. Changing `incomes[0]` later does not recalculate that stored value.

### 1(c) Independent copy and recalculated share

Here `saved_incomes` is copied before changing the first value. The share is calculated after the change, so it uses 1200.

In [4]:
incomes = [1000, 800, 0]
saved_incomes = incomes.copy()
incomes[0] = 1200
first_share = (incomes[0] - 800) / incomes[0]
saved_incomes.append(600)
print(incomes)
print(saved_incomes[-1], len(incomes))
print(f"{first_share:.0%}")

[1200, 800, 0]
600 3
33%


The three output lines after these changes are:

```text
[1200, 800, 0]
600 3
33%
```

`incomes.copy()` creates a separate list, so appending to `saved_incomes` leaves `incomes` at three elements. The recalculated share is `(1200 - 800) / 1200 = 1/3 = 33%`.

## Question 2: implement the policy as functions

Each function uses only the values passed to its parameters. The checks are performed before division or policy calculations.

In [5]:
def saving_share(income, consumption):
    """Return the saving share, or None when income is zero."""
    if income < 0 or consumption < 0:
        raise ValueError("income and consumption must be non-negative")

    if income == 0:
        return None

    return (income - consumption) / income


def transfer(income, consumption, size):
    """Return the monthly transfer under the assignment policy."""
    if income < 0 or consumption < 0 or size <= 0:
        raise ValueError("income and consumption must be non-negative and size must be positive")

    if income / size > 600:
        return 0

    shortfall = max(consumption - income, 0)
    cap = 100 * size
    return min(cap, shortfall)


def post_policy_share(income, consumption, size):
    """Return the saving share after applying the transfer policy."""
    support = transfer(income, consumption, size)
    income_after_support = income + support
    return saving_share(income_after_support, consumption)

**Design explanation:** Parameters are the names in a function definition, and arguments are the actual values supplied when calling it. `return` lets a function provide a result that other code can store, inspect, or pass to another function. Input checks come before division so invalid values raise `ValueError` instead of causing a calculation with invalid inputs or a division error. The helper functions are reused so the policy and share formulas have one implementation.

## Question 3: Build and interpret a checked report

The following cells use the functions from Question 2 and preserve the supplied input lists.

### 3(a) Prepare inputs

Convert each income string with an explicit loop. Check all four list lengths before processing so that no household can be silently skipped.

In [6]:
numeric_incomes = []
for income_string in income_text:
    numeric_incomes.append(float(income_string))

if not (len(household_ids) == len(numeric_incomes) == len(consumptions) == len(household_sizes)):
    raise ValueError("All four household input lists must have equal lengths")

print("Number of aligned households:", len(household_ids))

Number of aligned households: 8


### 3(b) Calculate household results

The loop builds a result list for each measure. Shares use percentage formatting; `None` is displayed as `undefined`.

In [7]:
transfers = []
savings_before = []
savings_after = []
shares_before = []
shares_after = []
incomes_after = []

for index in range(len(household_ids)):
    income = numeric_incomes[index]
    consumption = consumptions[index]
    size = household_sizes[index]

    support = transfer(income, consumption, size)
    income_after = income + support
    saving_before = income - consumption
    saving_after = income_after - consumption
    share_before = saving_share(income, consumption)
    share_after = saving_share(income_after, consumption)

    transfers.append(support)
    incomes_after.append(income_after)
    savings_before.append(saving_before)
    savings_after.append(saving_after)
    shares_before.append(share_before)
    shares_after.append(share_after)

def display_share(share):
    if share is None:
        return "undefined"
    return f"{share:.2%}"

print("ID   Income (EUR)  Consumption (EUR)  Size  Transfer (EUR)  Saving before (EUR)  Saving after (EUR)  Share before  Share after")
for index in range(len(household_ids)):
    print(f"{household_ids[index]:<4} {numeric_incomes[index]:>12.2f} {consumptions[index]:>18.2f} {household_sizes[index]:>5} "
          f"{transfers[index]:>15.2f} {savings_before[index]:>20.2f} {savings_after[index]:>19.2f} "
          f"{display_share(shares_before[index]):>12} {display_share(shares_after[index]):>12}")

ID   Income (EUR)  Consumption (EUR)  Size  Transfer (EUR)  Saving before (EUR)  Saving after (EUR)  Share before  Share after
A01       1200.00            1250.00     2           50.00               -50.00                0.00       -4.17%        0.00%
A02          0.00             150.00     1          100.00              -150.00              -50.00    undefined      -50.00%
A03       1500.00            1600.00     3          100.00              -100.00                0.00       -6.67%        0.00%
A04        700.00             850.00     1            0.00              -150.00             -150.00      -21.43%      -21.43%
A05       2400.00            2600.00     4          200.00              -200.00                0.00       -8.33%        0.00%
A06        800.00             950.00     2          150.00              -150.00                0.00      -18.75%        0.00%
A07          0.00               0.00     2            0.00                 0.00                0.00    undefined    u

### 3(c) Totals and averages

The pooled shares use total saving divided by total income. Household means include only defined shares, and the restricted comparison uses the same households in both periods.

In [8]:
total_transfer = 0
positive_transfer_count = 0
negative_saving_before_count = 0
negative_saving_after_count = 0
total_income_before = 0
total_income_after = 0
total_saving_before = 0
total_saving_after = 0
for index in range(len(household_ids)):
    total_transfer += transfers[index]
    if transfers[index] > 0:
        positive_transfer_count += 1
    if savings_before[index] < 0:
        negative_saving_before_count += 1
    if savings_after[index] < 0:
        negative_saving_after_count += 1
    total_income_before += numeric_incomes[index]
    total_income_after += incomes_after[index]
    total_saving_before += savings_before[index]
    total_saving_after += savings_after[index]

pooled_share_before = total_saving_before / total_income_before if total_income_before != 0 else None
pooled_share_after = total_saving_after / total_income_after if total_income_after != 0 else None

defined_before = []
defined_after = []
after_for_before_defined = []
for index in range(len(household_ids)):
    if shares_before[index] is not None:
        defined_before.append(shares_before[index])
        after_for_before_defined.append(shares_after[index])
    if shares_after[index] is not None:
        defined_after.append(shares_after[index])

mean_share_before = sum(defined_before) / len(defined_before) if defined_before else None
mean_share_after = sum(defined_after) / len(defined_after) if defined_after else None
mean_after_same_households = sum(after_for_before_defined) / len(after_for_before_defined) if after_for_before_defined else None

print(f"Total transfer: EUR {total_transfer:.2f}")
print(f"Households with positive transfer: {positive_transfer_count} / {len(household_ids)}")
print(f"Households with negative saving before: {negative_saving_before_count} / {len(household_ids)}")
print(f"Households with negative saving after: {negative_saving_after_count} / {len(household_ids)}")
print(f"Pooled saving share before: {display_share(pooled_share_before)} (total saving / total income)")
print(f"Pooled saving share after: {display_share(pooled_share_after)} (total saving / total income)")
print(f"Mean defined household share before: {display_share(mean_share_before)} ({len(defined_before)} households)")
print(f"Mean defined household share after: {display_share(mean_share_after)} ({len(defined_after)} households)")
print(f"Mean after, restricted to households with a defined pre-policy share: {display_share(mean_after_same_households)} ({len(after_for_before_defined)} households)")

Total transfer: EUR 600.00
Households with positive transfer: 5 / 8
Households with negative saving before: 6 / 8
Households with negative saving after: 2 / 8
Pooled saving share before: -4.76% (total saving / total income)
Pooled saving share after: 2.22% (total saving / total income)
Mean defined household share before: -6.19% (6 households)
Mean defined household share after: -7.03% (7 households)
Mean after, restricted to households with a defined pre-policy share: 0.13% (6 households)


**Interpretation:** The pooled share rises from about -4.76% to 2.22%, because aggregate saving rises by EUR 600 while aggregate income rises from EUR 8,400 to EUR 9,000. The arithmetic mean of defined household shares moves from about -6.19% to -7.03%. These means use different groups: the after-support mean adds A02, whose pre-support share was undefined because income was zero; its post-support share is -50%. Comparing different household sets can mix the policy change with a change in who is included. On the same six households with defined pre-support shares, the after-support mean is about 0.13%, compared with -6.19% before.

### 3(d) Sort defined pre-policy shares by distance to 0.10

The lambda receives one share from `sorted` and returns its absolute distance from the target. Sorting uses that distance as the key, while the resulting list still contains the original signed shares.

In [9]:
defined_pre_policy_shares = []
for share in shares_before:
    if share is not None:
        defined_pre_policy_shares.append(share)

target_share = 0.10
shares_nearest_to_target = sorted(
    defined_pre_policy_shares,
    key=lambda share: abs(share - target_share)
)
print("Defined pre-policy shares, nearest to 10% first:")
print([f"{share:.2%}" for share in shares_nearest_to_target])

Defined pre-policy shares, nearest to 10% first:
['22.22%', '-4.17%', '-6.67%', '-8.33%', '-18.75%', '-21.43%']


### 3(e) Counterfactual for A01

Copy the numeric income list, change only A01, and calculate the requested outcomes under both income values. The final comparison checks that the original numeric list was preserved.

In [10]:
original_numeric_incomes = numeric_incomes.copy()
counterfactual_incomes = numeric_incomes.copy()
counterfactual_incomes[0] = 1202

a01_original_income = numeric_incomes[0]
a01_counterfactual_income = counterfactual_incomes[0]
a01_original_transfer = transfer(a01_original_income, consumptions[0], household_sizes[0])
a01_counterfactual_transfer = transfer(a01_counterfactual_income, consumptions[0], household_sizes[0])
a01_original_income_after = a01_original_income + a01_original_transfer
a01_counterfactual_income_after = a01_counterfactual_income + a01_counterfactual_transfer
a01_original_saving_after = a01_original_income_after - consumptions[0]
a01_counterfactual_saving_after = a01_counterfactual_income_after - consumptions[0]

print("A01 counterfactual (EUR)")
print("Measure                 Original   Counterfactual   Change")
print(f"Original income        {a01_original_income:>8.2f} {a01_counterfactual_income:>15.2f} {a01_counterfactual_income - a01_original_income:>9.2f}")
print(f"Transfer               {a01_original_transfer:>8.2f} {a01_counterfactual_transfer:>15.2f} {a01_counterfactual_transfer - a01_original_transfer:>9.2f}")
print(f"Income after support   {a01_original_income_after:>8.2f} {a01_counterfactual_income_after:>15.2f} {a01_counterfactual_income_after - a01_original_income_after:>9.2f}")
print(f"Saving after support   {a01_original_saving_after:>8.2f} {a01_counterfactual_saving_after:>15.2f} {a01_counterfactual_saving_after - a01_original_saving_after:>9.2f}")

print("Original numeric income list unchanged:", numeric_incomes == original_numeric_incomes)

A01 counterfactual (EUR)
Measure                 Original   Counterfactual   Change
Original income         1200.00         1202.00      2.00
Transfer                  50.00            0.00    -50.00
Income after support    1250.00         1202.00    -48.00
Saving after support       0.00          -48.00    -48.00
Original numeric income list unchanged: True


**Interpretation:** A01 is eligible at EUR 1,200 because income per person is exactly EUR 600, so it receives EUR 50 and has EUR 1,250 income after support. At EUR 1,202, income per person is EUR 601, so the transfer falls to zero and income after support is EUR 1,202. Thus, a higher original income does not necessarily mean higher income after support: crossing the eligibility boundary removes support. Saving after support also changes from EUR 0 to EUR -48.

## Question 4: audit the proposed function

The proposed function is included for review. The diagnostic calls and their observed outcomes are shown in Markdown so they do not interrupt a run of the notebook.

In [11]:
def proposed_mean(incomes, consumptions):
    total = 0
    count = 0
    for income, consumption in zip(incomes, consumptions):
        share = saving_share(income, consumption)
        if share is not None and share != 0:
            total = 0
            total = total + share
            count = count + 1
    return total / count

### 4(a) Four distinct defects

1. `zip` stops at the shorter list, so unequal input lengths are silently accepted instead of raising `ValueError`.
2. The condition `share != 0` excludes valid zero shares, even though they belong in the arithmetic mean and its denominator.
3. `total = 0` is inside the loop, so each included share erases the accumulated total; with multiple nonzero shares, only the last one remains in the numerator.
4. The function always divides by `count`. When there are no defined shares (including empty inputs), `count` is zero and it raises `ZeroDivisionError` instead of returning `None`.

### 4(b) Repaired function

`mean_defined_share` first checks that the input lists have equal lengths. It then uses a loop and `saving_share`, includes zero shares, skips only undefined (`None`) shares, and returns `None` when no share is defined.

In [12]:
def mean_defined_share(incomes, consumptions):
    """Return the mean of defined saving shares, or None if there are none."""
    if len(incomes) != len(consumptions):
        raise ValueError("incomes and consumptions must have equal lengths")

    total = 0
    count = 0
    for income, consumption in zip(incomes, consumptions):
        share = saving_share(income, consumption)
        if share is not None:
            total += share
            count += 1

    if count == 0:
        return None
    return total / count

### 4(c) Diagnostic cases

| Inputs `(incomes, consumptions)` | Expected correct outcome | Proposal outcome | Defect exposed |
|---|---:|---:|---|
| `([100], [100])` | `0.0` | `ZeroDivisionError` | A valid zero share is excluded from the mean. |
| `([0], [10])` | `None` | `ZeroDivisionError` | No-defined-share case is not handled. |
| `([100, 200], [0])` | `ValueError` | `1.0` | Unequal lengths are silently truncated by `zip`. |
| `([100, 100], [0, 50])` | `0.75` | `0.25` | The running total is reset inside the loop. |

The first two cases both produce a division-by-zero error, but they test separate obligations: zero is a defined valid share, whereas `None` represents an undefined share.

Observed proposal calls and outcomes (shown here as Markdown, not executable cells):

```text
proposed_mean([100], [100])
ZeroDivisionError: division by zero
```

```text
proposed_mean([0], [10])
ZeroDivisionError: division by zero
```

```text
proposed_mean([100, 200], [0])
1.0
```

```text
proposed_mean([100, 100], [0, 50])
0.25
```

## Question 5: evidence and reproducible hand-in

### 5(a) Passing assertions

Run this cell after the preceding cells. Floating-point results are checked with `math.isclose`; undefined results are checked with `is None`. Four assertions include a note about the plausible mistake they catch.

In [13]:
# This catches a mistaken saving formula or a percentage result returned instead of a fraction.
assert math.isclose(saving_share(1000, 800), 0.20)

assert math.isclose(saving_share(1000, 1250), -0.25)
assert saving_share(0, 100) is None

# This catches a strict eligibility comparison that wrongly excludes exactly EUR 600 per person.
assert math.isclose(transfer(1200, 1300, 2), 100.0)

# This catches a missing or incorrectly calculated EUR 100-per-person transfer cap.
assert math.isclose(transfer(500, 1000, 2), 200.0)

assert math.isclose(transfer(500, 400, 2), 0.0)
assert mean_defined_share([0, 0], [50, 0]) is None

# This catches accidental mutation of the original numeric income list during the counterfactual.
assert numeric_incomes == original_numeric_incomes